In [26]:
import pandas as pd 
import geopandas as gpd
from pathlib import Path

sommarioni_p = Path('../../../1808_Sommarioni/')
sommarioni_geo_fp = list(sommarioni_p.rglob('sommarioni_geometries_202*.geojson'))[0]
sommarioni_txt_fp = list(sommarioni_p.rglob('sommarioni_text_data_with_pages_202*.json'))[0]
df = pd.read_json(sommarioni_txt_fp).sample(100)
gdf = gpd.read_file(sommarioni_geo_fp)
df['paradata'] = 'm'
df['start_time'] = "1808-01-01T00:00:00"
df['end_time'] = "1808-12-31T23:59:59"
df.drop(columns=['page', 'unique_id'], inplace=True)
gdf_star = gdf[gdf['geometry_id'].isin(df['geometry_id'])]
centre_gdf = gdf_star.dissolve(by='geometry_id').reset_index()[['geometry_id', 'geometry']]
centre_gdf['coordinate'] = centre_gdf['geometry'].apply(lambda v: v.centroid)
geometry_id_to_coordinate = centre_gdf.set_index('geometry_id')['coordinate'].to_dict()
df['coordinate'] = df['geometry_id'].apply(lambda v: geometry_id_to_coordinate[v])
column_order = ['unique_id','paradata', 'coordinate', 'start_time', 'end_time', 'geometry_id'] 
column_order = column_order + [c for c in df.columns if c not in column_order]
df = df.reset_index(drop=True).reset_index().rename(columns={'index': 'unique_id'})
df[column_order].to_csv('sommarioni_sample.csv', index=False)
gdf_star[['geometry_id', 'geometry']].to_file('sommarioni_sample.geojson', driver='GeoJSON')

indexable = [
    "place",
    "owner_standardised",
    "parcel_number",
    "sub_parcel_number",
]
name = 'Napoleonic Cadaster of 1808'
description = 'Digitisation of the 1808 Napoleonic cadaster of Venice. Contains both numeric facsimile of the cadaster map as well as their description in the written registry, called the \"Sommarioni\"'
paradata = 'The textual data from the register was manually transcribed, then semi-automatically cleaned and normalized by various researchers. The maps were vetcorized into geometries through computer vision techniques and then manually corrected and expanded.'
short_display = ['parcel_number', 'sub_parcel_number','owner_standardised', 'place', 'house_number', 'quality']
main_label = '${owner}'
sub_label = '${parcel_number} - {place}'
obj = {
    "name": name,
    "description": description,
    "paradata": paradata,
    "indexable": indexable,
    "short_display": short_display,
    "main_label": main_label,
    "sub_label": sub_label,
}
import json
with open('sommarioni_sample.json', 'w+') as f:
    json.dump(obj, f, indent=2)